# Fraud Mitigation Agent · Advanced 06 Automated Embeddings Atlas

Extensión opcional: prepara `autoEmbed` y consulta texto. Si la capacidad no está habilitada, se conserva el resultado de la ruta manual.


In [ ]:
import os, sys, json
from pathlib import Path

REPO_DIR = globals().get("REPO_DIR", "/content/fraud-mitigation-agent-workshop")
src = Path(REPO_DIR) / "src"
if src.exists() and str(src) not in sys.path:
    sys.path.insert(0, str(src))

from fraud_mitigation_agent.synthetic import seed_demo_data
from fraud_mitigation_agent.local import InMemoryDB

if "db" not in globals():
    from fraud_mitigation_agent.config import Settings
    from fraud_mitigation_agent.db import get_client, get_database
    settings = Settings.from_env()
    if settings.mongodb_uri:
        client = get_client(settings.mongodb_uri)
        db = get_database(client, settings.database_name)
    else:
        db = InMemoryDB()
seed_demo_data(db, reset=False)
print("Runtime listo:", type(db).__name__)


In [ ]:
from fraud_mitigation_agent.embeddings.automated_atlas import auto_embedding_index_definition, create_auto_embedding_index, automated_text_search
from fraud_mitigation_agent.tools.similarity import find_similar_fraud

print(json.dumps(auto_embedding_index_definition(), indent=2))
RUN_ATLAS_AUTO_EMBEDDINGS = False
if RUN_ATLAS_AUTO_EMBEDDINGS and hasattr(db.fraud_patterns, "create_search_index"):
    try:
        print(create_auto_embedding_index(db.fraud_patterns))
        rows = automated_text_search(db.fraud_patterns, "new device new ip impossible travel odd hour")
        print(json.dumps(rows, indent=2, default=str))
    except Exception as exc:
        print("Automated Embeddings no disponible; fallback manual:", exc)
else:
    tx = db.transactions.find_one({"tx_id": "tx-risky-001"}, {"_id": 0})
    print(json.dumps(find_similar_fraud(db, tx).data, indent=2, default=str))
